In [ ]:
# imports

import logging
import string
import json
from enum import Enum
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import typing
from numpy.typing import NDArray

In [ ]:
# slow import
import adaptive_polish.gis_measurement as gm
from adaptive_polish.dl_segmentation.sem_lamella_segmentor import SegmentationLabels
from adaptive_polish.strategy import AdaptivePolishMillingStrategy
from adaptive_polish.config import AdaptivePolishMillingConfig
from adaptive_polish.centring import get_bounding_box_scaled_to_image

In [ ]:
class NoImageFound(FileNotFoundError):
    pass


class NoExperimentFound(NoImageFound):
    pass

In [ ]:
# Constants
CONTINUE_FILENAME_STRING = "mill"
STOP_FILENAME_STRING = "stop"

base_path = (
    Path.home()
    / "OneDrive - The Rosalind Franklin Institute"
    / "Documents"
    / "test data"
    / "adaptive milling"
)

models = {
    "Gen0": (
        "0",
        base_path
        / "sem_models"
        / "Gen0"
        / "2024-02-24_0013_gis_lamela_crack_pytorch_AUnet.ptchkp",
    ),
    "Gen1 v2 Quality": (
        "1q",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v2"
        / "20250228_gen01_quality_1536_v2.pth",
    ),
    "Gen1 v3 Quality": (
        "1q",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v3"
        / "gen01_quality_1536_v3.pth",
    ),
    "Gen1 Performance": (
        "1p",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_performance_768"
        / "20250228_gen01_performance_768.pth",
    ),
    "Gen1 v3 Performance": (
        "1p",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_performance_768_v3"
        / "gen01_performance_768_v3.pth",
    ),
}
data_base_path = base_path / "2024segmentation_testdata"

In [ ]:
def get_array_and_pixel_size(image_path: Path) -> tuple[NDArray[typing.Any], float]:
    with tifffile.TiffFile(image_path) as tif:
        image_array = tif.asarray()
        # TODO: What's going on with the metadata?
        # pixel_size_m = tif.fei_metadata
        return image_array, tif.shaped_metadata[0]["pixel_size"]["x"]

In [ ]:
@dataclass
class TestImageInfo:
    path: Path
    data: NDArray[typing.Any]
    pixel_size_m: float
    expect_continue_milling: bool
    results: dict[str, typing.Any] = field(default_factory=dict)


class StrEnum(str, Enum):
    def __str__(self):
        return str(self.value)


class MillingDecision(StrEnum):
    Mill = "milling should continue"
    StopNoCleanPrediction = "no clean prediction"
    StopNoGISLayer = "no GIS layer detected"
    StopTooThinGISLayer = "minimum GIS thickness below threshold"
    StopCrackFound = "crack with area above threshold found"


class AdaptiveMillingTestResult(StrEnum):
    SUCCESS = "success"
    FAILED_TO_STOP = "failed to stop"
    FAILED_TO_CONTINUE = "failed to continue"

In [ ]:
# TODO: Use AdaptivePolishMillingStrategy directly
def process_SEM_image(
    model,
    config: AdaptivePolishMillingConfig,
    sem_image_info: TestImageInfo,
) -> tuple[MillingDecision, ...]:
    decision_reasons = []

    # Segmentation
    logging.info("Starting segmentation")
    prediction = model.predict(sem_image_info.data)
    logging.info("Segmentation complete")

    prediction_pixel_size_um = (
        sem_image_info.pixel_size_m
        * 1e6  # m to um
        * sem_image_info.data.shape[1]
        / prediction.shape[1]
    )

    mask_lamella_clean, mask_gis_clean, mask_crack_clean = gm.clean_prediction(
        prediction,
        additional_labels=(
            SegmentationLabels.GIS,
            SegmentationLabels.CRACK,
        ),
    )
    logging.info("Finished cleaning mask")

    if mask_gis_clean is None:
        logging.info("mask_gis_clean is None. Stopping")
        decision_reasons.append(MillingDecision.StopNoCleanPrediction)

    if not np.sum(mask_lamella_clean):
        logging.info("mask_lamella_clean is None. Stopping")
        decision_reasons.append(MillingDecision.StopNoCleanPrediction)
    # Is there any GIS?
    if not np.any(mask_gis_clean):
        # No GIS detected
        logging.info("No GIS layer detected in mask_gis_clean. Stopping")
        decision_reasons.append(MillingDecision.StopNoGISLayer)
    else:
        # Get lamella bounding box
        lamella_bbox = get_bounding_box_scaled_to_image(
            sem_image_info.data, mask=mask_lamella_clean
        )

        # Measure GIS
        # Measure GIS
        gis_thickness_px = np.sum(
            gm.resize_image(mask_gis_clean, new_shape=sem_image_info.data.shape),
            axis=0,
        )

        gis_thickness_um = gis_thickness_px * sem_image_info.metadata.pixel_size.x * 1e6

        gis_xlims_px = np.asarray(
            (
                np.argmax(gis_thickness_px > 1),
                len(gis_thickness_px) - 1 - np.argmax(gis_thickness_px[::-1] > 1),
            )
        )

        lamella_xlims_px = np.round(
            (
                lamella_bbox[1],
                lamella_bbox[3],
            )
        ).astype(np.uint32)

        maximum_side_difference_px = int(
            round(
                config.maximum_side_difference_um
                / (sem_image_info.metadata.pixel_size.x * 1e6)
            )
        )

        # Allow maximum of maximum_side_difference_um inward from lamella edge
        xlims_px = (
            min(gis_xlims_px[0], lamella_xlims_px[0] + maximum_side_difference_px),
            max(gis_xlims_px[1], lamella_xlims_px[1] - maximum_side_difference_px),
        )

        gis_thickness_filtered_um = gm.filter_gis_thickness(
            gis_thickness_um[xlims_px[0] : xlims_px[1] + 1],
            window_size_m=10 * sem_image_info.pixel_size_m,
            pixel_size_m=sem_image_info.pixel_size_m,
        )

        gis_thickness_filtered_um = gis_thickness_um[xlims_px[0] : xlims_px[1] + 1]
        min_gis_um = np.nanmin(gis_thickness_filtered_um)
        logging.info(f"Took {len(gis_thickness_filtered_um)} GIS measurements along x")
        logging.info(
            "Minimum GIS thickness for milling cycle %s = %f um",
            sem_image_info.path,
            min_gis_um,
        )

        # Should we continue?
        if min_gis_um < config.gis_stop_um:
            logging.info(
                f"Stopping as minimum GIS (um) {min_gis_um} < threshold {config.gis_stop_um}"
            )
            decision_reasons.append(MillingDecision.StopTooThinGISLayer)

        # Crack detection
        if mask_crack_clean is None:
            crack_area_um2 = 0
        else:
            crack_area_um2 = gm.get_mask_area_um2(
                mask_crack_clean, pixel_size_um=prediction_pixel_size_um
            )

        logging.info(
            "Area of cracks found in %s = %f um2", sem_image_info.path, crack_area_um2
        )

        if crack_area_um2 > config.max_crack_area_um2:
            logging.info(
                f"Stopping as crack area (um2) {crack_area_um2} > threshold {config.max_crack_area_um2}"
            )
            decision_reasons.append(MillingDecision.StopCrackFound)

        pad_width = np.asarray(
            (xlims_px[0], len(gis_thickness_um) - xlims_px[1]), dtype=np.int32
        )

        sem_image_info.results["gis_um"] = np.pad(
            # Pad with NaNs so that plot works correctly
            gis_thickness_filtered_um,
            pad_width=pad_width,
            constant_values=np.nan,
        )
        sem_image_info.results["min_gis_um"] = min_gis_um
        sem_image_info.results["xlims"] = xlims_px
        sem_image_info.results["crack_area_um2"] = crack_area_um2

    sem_image_info.results["prediction"] = prediction
    sem_image_info.results["mask_gis_clean"] = mask_gis_clean

    if not decision_reasons:
        decision_reasons.append(MillingDecision.Mill)

    return tuple(decision_reasons)

In [ ]:
def process_images(
    model,
    config: AdaptivePolishMillingConfig,
    sem_image_info: TestImageInfo,
    fib_image_info: TestImageInfo,
    plot_dir: Path,
    plot_basename: str,
) -> tuple[MillingDecision, ...]:
    stop_milling_reasons = process_SEM_image(model, config, sem_image_info)
    logging.info(f"Saving {plot_basename}_plot.png")
    gis_um = sem_image_info.results.get("gis_um")
    gm.milling_cycle_plot(
        sem_image=sem_image_info.data,
        first_prediction=sem_image_info.results["prediction"],
        clean_prediction=sem_image_info.results["mask_gis_clean"],
        fib_image=fib_image_info.data,
        gis_thickness_um=gis_um,
        gis_stop_um=config.gis_stop_um,
        crack_area_um2=sem_image_info.results.get("crack_area_um2"),
        img_name=plot_basename,
        fib_screenshot=None,
        save_path=plot_dir / f"{plot_basename}_plot.png",
    )
    # if gis_m is not None:
    #     xlims = sem_image_info.results["xlims"]
    #     PIL.Image.fromarray(
    #         bitmaps.create_bitmap_array(
    #             sem_image_info.results.get("GIS_m"),
    #             xlims=xlims,
    #             window_size_m=10,
    #             gis_stop_m=gis_stop_m,
    #             gis_full_power_m=gis_full_power_m,
    #             dtype=np.uint8,
    #         )
    #     ).save(plot_dir / f"{plot_basename}_mill.bmp", format="BMP")
    return stop_milling_reasons


def get_step_info(
    directory_path: Path, index: int, step_char: str, image_type: str
) -> TestImageInfo:
    file_stem = f"{index:02d}{step_char}_{image_type}"
    continue_path = directory_path / f"{file_stem}_{CONTINUE_FILENAME_STRING}.tif"
    stop_path = directory_path / f"{file_stem}_{STOP_FILENAME_STRING}.tif"
    if continue_path.is_file():
        image_path = continue_path
        expect_continue = True
    elif stop_path.is_file():
        image_path = stop_path
        expect_continue = False
    else:
        raise NoImageFound(f"No image found for {file_stem}")

    image_array, pixel_size_m = get_array_and_pixel_size(image_path)

    return TestImageInfo(image_path, image_array, pixel_size_m, expect_continue)

In [ ]:
def process_experiment_step(
    model,
    config: AdaptivePolishMillingConfig,
    fib_path: Path,
    sem_path: Path,
    plot_dir: Path,
    index: int,
    step_char: str,
) -> tuple[AdaptiveMillingTestResult, tuple[MillingDecision, ...]]:
    sem_image_info = get_step_info(sem_path, index, step_char, "SEM")
    fib_image_info = get_step_info(fib_path, index, step_char, "FIB")
    assert (
        sem_image_info.expect_continue_milling == fib_image_info.expect_continue_milling
    ), "Milling continue or stop does not match between SEM and FIB images"

    milling_decisions = process_images(
        model,
        config,
        sem_image_info,
        fib_image_info,
        plot_dir,
        f"{index:02d}{step_char}",
    )
    continue_milling = milling_decisions[0] == MillingDecision.Mill
    if sem_image_info.expect_continue_milling:
        if continue_milling:
            return AdaptiveMillingTestResult.SUCCESS, milling_decisions
        else:
            logging.warning(
                "Milling failed to continue for %02d%s (expected continue=%s), decision(s): %s",
                index,
                step_char,
                sem_image_info.expect_continue_milling,
                ", ".join(str(_) for _ in milling_decisions),
            )
            return AdaptiveMillingTestResult.FAILED_TO_CONTINUE, milling_decisions
    else:
        if continue_milling:
            logging.warning(
                "Milling failed to stop for %02d%s (expected continue=%s)",
                index,
                step_char,
                sem_image_info.expect_continue_milling,
            )
            return AdaptiveMillingTestResult.FAILED_TO_STOP, milling_decisions
        else:
            return AdaptiveMillingTestResult.SUCCESS, milling_decisions

In [ ]:
def process_experiment(
    model,
    config: AdaptivePolishMillingConfig,
    fib_path: Path,
    sem_path: Path,
    plot_dir: Path,
    index: int,
) -> tuple[tuple[AdaptiveMillingTestResult, tuple[MillingDecision, ...]], ...]:
    outputs: list[AdaptiveMillingTestResult, tuple[MillingDecision, ...]] = []
    step_characters = string.ascii_uppercase
    for step_char in step_characters:
        try:
            outputs.append(
                process_experiment_step(
                    model, config, fib_path, sem_path, plot_dir, index, step_char
                )
            )
        except NoImageFound:
            if step_char == step_characters[0]:
                # Only raise an error if it is the first file in the series
                raise NoExperimentFound(f"No experiment {index} found")
            break
    return tuple(outputs)

In [ ]:
model_key = "Gen1 v3 Quality"
model_generation, model_path = models[model_key]
config = AdaptivePolishMillingConfig(
    model_path=model_path,
    model_generation=model_generation,
    window_size_px=10,
    max_crack_area_um2=0.5,
    gis_stop_um=0.25,
    align_sem=False,
    # This is new to deal with the lamella edge and GIS edge not being segmented to quite the same X:
    maximum_side_difference_um=0.05,
)
output_base_path = data_base_path / model_key
if output_base_path.is_dir():
    logging.warning("%s already exists, data may be overwritten", output_base_path)

output_base_path.mkdir(exist_ok=True)

plot_dir = output_base_path / "plots"
plot_dir.mkdir(exist_ok=True)

results_json_path = output_base_path / "results.json"
results_pie_plots_path = output_base_path / "results_pie_plots.png"
results_bar_plots_path = output_base_path / "results_bar_plots.png"


In [ ]:
logging.info("Initialising DL model, path: %s", config.model_path)
model = gm.load_sem_model(config.model_path, generation=config.model_generation)

In [ ]:
# iterate
i = 1
results: dict[
    int, tuple[tuple[AdaptiveMillingTestResult, tuple[MillingDecision, ...]], ...]
] = {}

while True:
    try:
        results[i] = process_experiment(
            model, config, data_base_path / "FIB", data_base_path / "SEM", plot_dir, i
        )
        i += 1
    except NoExperimentFound:
        logging.info("Stopping processing as no experiment was found for %i", i)
        break
    except Exception:
        logging.error("Unexpected error", exc_info=True)
        raise

In [ ]:
with results_json_path.open("w+") as _:
    json.dump(results, _)

In [ ]:
# Can be run to avoid rerunning previous bits:
with results_json_path.open("r") as _:
    results: dict[
        int, tuple[tuple[AdaptiveMillingTestResult, tuple[MillingDecision, ...]], ...]
    ] = json.load(_)

In [ ]:
def create_pie_plots(
    results: dict[
        int, tuple[tuple[AdaptiveMillingTestResult, tuple[MillingDecision, ...]], ...]
    ],
    output_path: Path,
) -> None:
    # Generate stats for plots
    experiment_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}

    image_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}
    failures_per_experiment: dict[int, int] = {}
    keeps_failing_to_stop: dict[str, int] = {
        "never stops": 0,
        "stops later": 0,
        "no further images": 0,
    }
    good_decisions: dict[MillingDecision, int] = {str(_): 0 for _ in MillingDecision}
    bad_decisions: dict[MillingDecision, int] = {str(_): 0 for _ in MillingDecision}

    total_experiments = len(results.keys())
    total_images = sum((len(_) for _ in results.values()))

    print(f"Total number of experiments: {total_experiments}")
    print(f"Total number of images: {total_images}")

    for experiment_idx, experiment_outputs in results.items():
        # Get experiment success
        experiment_results = tuple(_[0] for _ in experiment_outputs)
        if str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE) in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE)] += 1

        elif AdaptiveMillingTestResult.FAILED_TO_STOP in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_STOP)] += 1

            # Check if failing continues after an initial FAILED_TO_STOP
            first_failed_to_stop = experiment_results.index(
                str(AdaptiveMillingTestResult.FAILED_TO_STOP)
            )
            if first_failed_to_stop != len(experiment_results) - 1:
                if len(set(experiment_results[first_failed_to_stop:])) == 1:
                    keeps_failing_to_stop["never stops"] += 1
                else:
                    keeps_failing_to_stop["stops later"] += 1
            else:
                keeps_failing_to_stop["no further images"] += 1

        else:
            experiment_success[str(AdaptiveMillingTestResult.SUCCESS)] += 1

        # Get image success
        failure_count = 0
        for outputs in experiment_outputs:
            image_success[str(outputs[0])] += 1
            if outputs[0] == AdaptiveMillingTestResult.SUCCESS:
                key = " & ".join(outputs[1])
                good_decisions[key] = good_decisions.get(key, 0) + 1
            else:
                failure_count += 1
                key = " & ".join(outputs[1])
                bad_decisions[key] = bad_decisions.get(key, 0) + 1

        # Count how many times failed experiments failed
        failures_per_experiment[failure_count] = (
            failures_per_experiment.get(failure_count, 0) + 1
        )

    fig, axes = plt.subplots(2, 3, figsize=(20, 14), dpi=300)

    fig.suptitle("Results of SEM GIS detection\n\n", size="x-large")

    axes[0, 0].pie(
        experiment_success.values(),
        labels=tuple(str(_) if _ > 0 else "" for _ in experiment_success.values()),
    )
    axes[0, 0].legend(experiment_success.keys(), loc="best")
    axes[0, 0].set_title("Experiment results")

    axes[0, 1].pie(
        failures_per_experiment.values(),
        labels=failures_per_experiment.values(),
    )
    axes[0, 1].legend(
        tuple(f"Failures: {_}" for _ in failures_per_experiment.keys()), loc="best"
    )
    axes[0, 1].set_title("Number of failures per experiment")

    axes[0, 2].pie(
        keeps_failing_to_stop.values(), labels=keeps_failing_to_stop.values()
    )
    axes[0, 2].legend(keeps_failing_to_stop.keys(), loc="best")
    axes[0, 2].set_title("If an experiment fails to stop, what happens next?")

    axes[1, 0].pie(
        image_success.values(),
        labels=tuple(str(_) if _ > 0 else "" for _ in image_success.values()),
    )
    axes[1, 0].legend(image_success.keys(), loc="best")
    axes[1, 0].set_title("Image results")

    axes[1, 1].pie(
        good_decisions.values(),
        labels=tuple(str(_) if _ > 0 else "" for _ in good_decisions.values()),
        shadow=False,
    )
    axes[1, 1].legend(good_decisions.keys(), loc="best")
    axes[1, 1].set_title("Good image decision reasons")

    axes[1, 2].pie(
        bad_decisions.values(),
        labels=bad_decisions.values(),
    )
    axes[1, 2].legend(bad_decisions.keys(), loc="best")
    axes[1, 2].set_title("Bad image decision reasons")

    fig.tight_layout()
    fig.savefig(output_path)
    fig.show()

In [ ]:
def create_bar_plots(
    results: dict[
        int, tuple[tuple[AdaptiveMillingTestResult, tuple[MillingDecision, ...]], ...]
    ],
    output_path: Path,
) -> None:
    # Generate stats for plots
    experiment_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}

    image_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}
    failures_per_experiment: dict[int, int] = {}
    keeps_failing_to_stop: dict[str, int] = {
        "never stops": 0,
        "stops later": 0,
        "no further images": 0,
    }
    good_decisions: dict[MillingDecision, int] = {str(_): 0 for _ in MillingDecision}
    bad_decisions: dict[MillingDecision, int] = {str(_): 0 for _ in MillingDecision}

    total_experiments = len(results.keys())
    total_images = sum((len(_) for _ in results.values()))

    print(f"Total number of experiments: {total_experiments}")
    print(f"Total number of images: {total_images}")

    for experiment_idx, experiment_outputs in results.items():
        # Get experiment success
        experiment_results = tuple(_[0] for _ in experiment_outputs)
        if str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE) in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE)] += 1

        elif AdaptiveMillingTestResult.FAILED_TO_STOP in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_STOP)] += 1

            # Check if failing continues after an initial FAILED_TO_STOP
            first_failed_to_stop = experiment_results.index(
                str(AdaptiveMillingTestResult.FAILED_TO_STOP)
            )
            if first_failed_to_stop != len(experiment_results) - 1:
                if len(set(experiment_results[first_failed_to_stop:])) == 1:
                    keeps_failing_to_stop["never stops"] += 1
                else:
                    keeps_failing_to_stop["stops later"] += 1
            else:
                keeps_failing_to_stop["no further images"] += 1

        else:
            experiment_success[str(AdaptiveMillingTestResult.SUCCESS)] += 1

        # Get image success
        failure_count = 0
        for outputs in experiment_outputs:
            image_success[str(outputs[0])] += 1
            if outputs[0] == AdaptiveMillingTestResult.SUCCESS:
                key = " &\n".join(outputs[1])
                good_decisions[key] = good_decisions.get(key, 0) + 1
            else:
                failure_count += 1
                key = " &\n".join(outputs[1])
                bad_decisions[key] = bad_decisions.get(key, 0) + 1

        # Count how many times failed experiments failed
        failures_per_experiment[failure_count] = (
            failures_per_experiment.get(failure_count, 0) + 1
        )

    fig, axes = plt.subplots(2, 3, figsize=(20, 14), dpi=300)

    fig.suptitle("Results of SEM GIS detection\n\n", size="x-large")

    _ = axes[0, 0].barh(
        experiment_success.keys(),
        experiment_success.values(),
        align="center",
    )
    axes[0, 0].bar_label(_, padding=2)
    axes[0, 0].set_xlim(0, total_experiments)
    axes[0, 0].set_title("Experiment results")

    _ = axes[0, 1].barh(
        tuple(str(_) for _ in failures_per_experiment.keys()),
        failures_per_experiment.values(),
        align="center",
    )
    axes[0, 1].bar_label(_, padding=2)
    axes[0, 1].set_xlim(0, total_experiments)
    axes[0, 1].set_title("Number of failures per experiment")

    _ = axes[0, 2].barh(
        keeps_failing_to_stop.keys(), keeps_failing_to_stop.values(), align="center"
    )
    axes[0, 2].bar_label(_, padding=2)
    axes[0, 2].set_xlim(0, sum(keeps_failing_to_stop.values()))
    axes[0, 2].set_title("If an experiment fails to stop, what happens next?")

    _ = axes[1, 0].barh(image_success.keys(), image_success.values(), align="center")
    axes[1, 0].bar_label(_, padding=2)
    axes[1, 0].set_xlim(0, total_images)
    axes[1, 0].set_title("Image results")

    _ = axes[1, 1].barh(
        tuple(str(_) for _ in good_decisions.keys()),
        good_decisions.values(),
        align="center",
    )
    axes[1, 1].bar_label(_, padding=2)
    axes[1, 1].set_xlim(0, sum(good_decisions.values()))
    axes[1, 1].set_title("Good image decision reasons")

    _ = axes[1, 2].barh(
        tuple(str(_) for _ in bad_decisions.keys()),
        bad_decisions.values(),
        align="center",
    )
    axes[1, 2].bar_label(_, padding=2)
    axes[1, 2].set_xlim(0, sum(bad_decisions.values()))
    axes[1, 2].set_title("Bad image decision reasons")

    fig.tight_layout()
    fig.savefig(output_path)
    fig.show()

In [ ]:
create_pie_plots(results, output_path=results_pie_plots_path)

In [ ]:
create_bar_plots(results, output_path=results_bar_plots_path)